# Session 2 — Automatic differentiation and nonconvex optimisation**M2 MACIA · Physics-Informed Neural Networks for PDEs**Paul Boureau — AGM, CY Cergy Paris University---### How this notebook worksIt alternates between two kinds of question.> **T** — **theory.** Paper and pencil. You think, we correct **at the board** together.> Nothing to run. Write your answer in the cell provided if you want a record of it.> **C** — **code.** You write it, here, yourself. The cells are scaffolded: the boring> parts are given, the lines that carry the idea are marked `# TODO`.Every **C** answers a question asked by the **T** before it. That is the whole design:you prove a statement, then you measure it ten minutes later.**Run the setup cell below first.** Then work in order — Part 4 uses nothing fromParts 1–3, but it will not mean much without them.

In [ ]:
import timeimport numpy as npimport torchimport matplotlib.pyplot as plttorch.set_default_dtype(torch.float64)   # this line is itself studied in Part 2torch.manual_seed(0); np.random.seed(0)TEAL, WINE, OCHRE = "#0E6A62", "#8A3A55", "#7E5C0E"def timeit(f, reps=200, warm=50):    """Average time of one call to f, after a warm-up.    The warm-up is not decoration: the first calls pay for memory allocation and    kernel selection, and are not representative. Part 3 asks you to remove it    and look at what happens.    """    for _ in range(warm): f()    t0 = time.perf_counter()    for _ in range(reps): f()    return (time.perf_counter() - t0) / repsprint("torch", torch.__version__, "| dtype", torch.get_default_dtype())

---# Part 1 — How does a computer differentiate a network?## T1 — theory *(corrected at the board)*Take the smallest possible network: one input $x$, three parameters $w, b, c$,$$y \;=\; c\,\tanh(wx+b), \qquad x = 2,\quad w = 0.5,\quad b = -0.3,\quad c = 1.5 .$$**T1.1** Write the **evaluation trace** of this computation: the list of elementaryoperations, each using only results already available. Use the names$v_{-2} = w$, $v_{-1} = b$, $v_0 = c$ for the inputs and $v_1, v_2, v_3, v_4$ for theintermediate results. Give the numerical value of each $v_i$.**T1.2** *(forward mode)* Differentiate the trace line by line, **top to bottom**, in thedirection $(\dot w, \dot b, \dot c) = (1, 0, 0)$: that is, set $\dot v_i :=\partial v_i / \partial w$ and propagate. Give $\partial y/\partial w$.Check it against the closed-form derivative.**T1.3** How many such sweeps do you need to obtain **all three** partial derivatives?And for a network with $P$ parameters?**T1.4** *(reverse mode)* Now define the **adjoint** $\bar v_i := \partial y / \partial v_i$— a derivative of the *output*, not of an input. Seed $\bar v_4 = 1$ and go **bottom totop**, using$$\bar v_i \;=\; \sum_{j \,:\, i \to j} \bar v_j \, \frac{\partial v_j}{\partial v_i}.$$Give $\partial y/\partial c$, $\partial y/\partial b$, $\partial y/\partial w$.How many sweeps did that take?**T1.5** State, in one sentence, where the asymmetry between the two modes comes from.*(Hint: count inputs and outputs.)***T1.6** Reverse mode needs something forward mode does not. What, and what does it cost?**T1.7** Theorem 2.8 of the notes (Baur–Strassen) says$\mathrm{cost}(\nabla F) \le C\,\mathrm{cost}(F)$ with $C$ **independent of $P$**.Before running anything: what do you expect the ratio$\mathrm{cost}(\text{forward} + \text{gradient}) / \mathrm{cost}(\text{forward})$ to do as$P$ grows from $300$ to $10^6$? **Write a number down.**

*Your notes on T1 — double-click to edit.*

## C1 — code: is the constant really blind to $P$?Four two-hidden-layer $\tanh$ networks of increasing width, evaluated on the same 512points. Time one forward pass and one forward-plus-gradient, and print the ratio.

In [ ]:
def mlp(x, Ps):    h = x    for W, b in Ps[:-1]:        h = torch.tanh(h @ W + b)    W, b = Ps[-1]    return (h @ W + b).squeeze()def make(width):    """Build a 1 -> width -> width -> 1 network. Returns (layers, flat list, P)."""    Ps = [(torch.randn(1, width),     torch.randn(width)),          (torch.randn(width, width), torch.randn(width)),          (torch.randn(width, 1),     torch.randn(1))]    flat = [t for p in Ps for t in p]    for t in flat:        t.requires_grad_(True)    return Ps, flat, sum(t.numel() for t in flat)x = torch.linspace(-1, 1, 512).reshape(-1, 1)print(f"{'width':>6} {'P':>9} {'forward (ms)':>13} {'fwd+grad (ms)':>14} {'ratio':>7}")for width in [16, 64, 256, 1024]:    Ps, flat, P = make(width)    # TODO (1) time one forward pass, loss = mlp(x, Ps).pow(2).mean()    #          use timeit(lambda: ...)    # TODO (2) time one forward pass FOLLOWED BY its gradient, with    #          torch.autograd.grad(loss, flat)    # TODO (3) print width, P, both timings in ms, and their ratio    raise NotImplementedError("C1")

**T1.8** *(to discuss)* $P$ was multiplied by about $3\,300$. What did the ratio do?Compare with the number you wrote down in T1.7.**T1.9** The ratio is **not** monotone — it is largest for the *smallest* network. Explain.Which rows of the table actually test Theorem 2.8?

---# Part 2 — Why not just use finite differences?## T2 — theory *(corrected at the board)*Write $\varepsilon_M$ for the unit roundoff ($\approx 2.2\cdot10^{-16}$ in double precision),$M_k := \sup |f^{(k)}|$ near $x$, and $M_0 := \sup|f|$.**T2.1** For the central difference $D_h f(x) = \dfrac{f(x+h)-f(x-h)}{2h}$, use Taylor to show$$D_h f(x) \;=\; f'(x) + \frac{h^2}{6} f'''(x) + O(h^4),$$so that the **truncation** error is at most $\tfrac{h^2}{6} M_3$.**T2.2** Each evaluation is computed as $f(x)(1+\delta)$ with $|\delta| \le \varepsilon_M$.Show that the **roundoff** contribution to $D_h f$ is at most $\varepsilon_M M_0 / h$.**T2.3** Minimise $E(h) = \tfrac{h^2}{6}M_3 + \dfrac{\varepsilon_M M_0}{h}$ over $h > 0$.Give $h^\star$ and $E(h^\star)$, and the power of $\varepsilon_M$ in each.**T2.4** Do the same for the second derivative:$$D^2_h f(x) = \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}.$$Show the truncation error is at most $\tfrac{h^2}{12}M_4$ and the roundoff at most$4\varepsilon_M M_0/h^2$ *(mind the coefficients $1, 2, 1$)*. Minimise. What power of$\varepsilon_M$ do you get now?**T2.5** Put numbers in: in double precision, what is the best relative accuracy you can hopefor on $f'$? On $f''$? And in single precision, $\varepsilon_M \approx 1.2\cdot10^{-7}$?**T2.6** A PINN residual for a second-order PDE *is* a second derivative, evaluated at everycollocation point at every step. What does T2.5 say about building it from finite differences?Say precisely which of the two error terms of Session 1 this floor belongs to — or whether itbelongs to neither.

*Your notes on T2 — double-click to edit.*

## C2 — code: measure the two floorsTake $f(x) = \tanh(3x + \tfrac12)$ at $x_0 = 0.7$. Automatic differentiation gives the twoexact derivatives; sweep $h$ and watch the two quotients fail.

In [ ]:
# exact values, by automatic differentiation — no h anywherex0 = torch.tensor(0.7, requires_grad=True)d1, = torch.autograd.grad(torch.tanh(3*x0 + 0.5), x0, create_graph=True)d2, = torch.autograd.grad(d1, x0)D1, D2 = d1.item(), d2.item()print(f"autograd   f'  = {D1:.16f}")print(f"autograd   f'' = {D2:.16f}")f  = lambda z: np.tanh(3*z + 0.5)xf = 0.7es = np.arange(-0.5, -9.01, -0.25)        # h = 10**e# TODO (1) for each e in es, with h = 10.0**e, compute#            fd1 = (f(xf+h) - f(xf-h)) / (2*h)#            fd2 = (f(xf+h) - 2*f(xf) + f(xf-h)) / h**2#          and store the RELATIVE errors |fd1-D1|/|D1| and |fd2-D2|/|D2|.## TODO (2) plot both, semilogy, against es. Add horizontal lines at#          eps**(2/3) and eps**(1/2), with eps = np.finfo(float).eps.## TODO (3) print the smallest error of each curve and the h where it occurs.## TODO (4) compare those two h with the h* you computed in T2.3 and T2.4.#          For this f:  M0 = |f(x0)|,  M3 = 27*sup|tanh'''| = 54,#                       M4 = 81*sup|tanh''''| = 330.96.#          Does the formula locate the minimum? Does it predict its height?raise NotImplementedError("C2")

**T2.7** *(to discuss)* Two things to read separately in that output.* The **location** of each minimum, against the $h^\star$ of T2.3 and T2.4. How close?* The **height** of each minimum, against the bound evaluated at $h^\star$. The bound sits  about two orders of magnitude *above* what you measured. Is the proposition wrong? If not,  which step of its proof is responsible for the slack?## C2b — and now in single precisionOne line to change. This is the number to remember from Part 2.

In [ ]:
# TODO: redo the f'' sweep in float32 — i.e. cast xf, h and every evaluation with#       np.float32 — and report the new floor. Compare with eps32**(1/2), where#       eps32 = np.finfo(np.float32).eps.##       Then answer: a PINN assembling its residual by finite differences in single#       precision carries a floor of what size, before any training at all?raise NotImplementedError("C2b")

---# Part 3 — Differentiating in $x$: what a Laplacian costs## T3 — theory *(corrected at the board)*Let $\Phi_\theta : \mathbb{R}^d \to \mathbb{R}$ and $H(x) = \nabla_x^2 \Phi_\theta(x)$.**T3.1** Explain why a **Hessian–vector product** $v \mapsto H v$ can be computed at cost$O(\mathrm{cost}(\Phi))$, by applying forward mode to the function $x \mapsto \nabla_x \Phi_\theta(x)$produced by a reverse sweep. Why is that cost independent of $d$?**T3.2** The Laplacian is a trace:$$\Delta_x \Phi_\theta(x) \;=\; \operatorname{tr} H(x) \;=\; \sum_{i=1}^{d} e_i^{\top} H(x)\, e_i .$$What does it cost, in Hessian–vector products? Is that an inefficiency of the library?**T3.3** *(Hutchinson)* Let $v$ be random with $\mathbb{E}[vv^{\top}] = I$. Prove, in threeequalities, that for **every** matrix $A$$$\mathbb{E}\big[v^{\top} A v\big] \;=\; \operatorname{tr} A .$$*(Cyclicity of the trace, linearity, hypothesis.)***T3.4** For Rademacher $v$ (entries $\pm 1$ with probability $\tfrac12$) and symmetric $A$,$$\operatorname{Var}\big(v^{\top} A v\big) \;=\; 2\Big(\|A\|_F^2 - \sum_i A_{ii}^2\Big).$$Which operators make this small? Which make it large? Can you tell in advance which case youare in? And: replacing $\Delta\Phi_\theta$ by an unbiased estimate changes *what is beingminimised* — say precisely how.**T3.5** Before running C3: going from $d = 2$ to $d = 50$, by what factor does the cost of**one** HVP change? And of the **full** Laplacian? **Write both numbers down.**

*Your notes on T3 — double-click to edit.*

## C3 — code: pay the billA width-64 network, one input point, $d \in \{2, 10, 50\}$. Express every cost as amultiple of one forward pass.

In [ ]:
def make_net(d, n=64):    W1, b1 = torch.randn(d, n), torch.randn(n)    W2, b2 = torch.randn(n, n), torch.randn(n)    W3, b3 = torch.randn(n, 1), torch.randn(1)    def f(z):        h = torch.tanh(z @ W1 + b1)        h = torch.tanh(h @ W2 + b2)        return (h @ W3 + b3).squeeze()    return fprint(f"{'d':>4} {'forward (ms)':>13} {'1 HVP':>10} {'Laplacian':>12} {'Lap/HVP':>9}")for d in [2, 10, 50]:    f = make_net(d)    x = torch.randn(1, d, requires_grad=True)    I = torch.eye(d)    # TODO (1) write hvp(v) returning H @ v, in two steps:    #            y = f(x);  g, = torch.autograd.grad(y, x, create_graph=True)    #            then differentiate (g*v).sum() with respect to x    #    # TODO (2) time one forward pass, one hvp, and the full Laplacian    #            sum(hvp(I[i:i+1])[0, i] for i in range(d))    #          Use the SAME reps and warm on all three — this matters, see C3b.    #    # TODO (3) print d, the forward time, and the two costs as multiples of a    #          forward pass, plus the ratio Laplacian / HVP.    raise NotImplementedError("C3")

**T3.6** *(to discuss)* What **should** the last column contain? Does it? Read the secondcolumn too: what is it saying, and is it the same kind of statement?**T3.7** Price one second-order collocation point in $d = 100$, in forward passes, using yourown measurements. Session 4 will claim that networks have a theoretical advantage in highdimension — reconcile the two.## C3b — exact values, inexact costsHere is a true story about the cell you have just written.> An earlier version of it timed the three quantities with different warm-ups, and reported,> at $d = 2$, a Laplacian **cheaper than a single Hessian–vector product**.**T3.8** At $d = 2$ the Laplacian *is* two Hessian–vector products. So that row was impossible.(a) Say why, in one line. (b) What could produce it? (c) What rule does it suggest?The cell below separates the two things that were confused there: the **value**, which isexact, and the **cost**, which is a measurement.

In [ ]:
d = 10f = make_net(d)x = torch.randn(1, d, requires_grad=True)I = torch.eye(d)def hvp(v):    y = f(x)    g, = torch.autograd.grad(y, x, create_graph=True)    return torch.autograd.grad((g*v).sum(), x)[0]# TODO (1) THE VALUE. Compute the Laplacian in two independent ways:#            - as the trace, sum(hvp(I[i:i+1])[0, i] for i in range(d))#            - by assembling the full Hessian row by row and taking torch.trace#          Print both and their difference. How many digits agree?## TODO (2) THE COST. Repeat the timing of C3 five times in a row, at this same d,#          and print HVP, Laplacian and their ratio each time.#          The ratio should be d. How much does it actually move?## TODO (3) Conclude: which of the two quantities above is a number, and which is a#          random variable? Which digits of the table in C3 were you entitled to read?raise NotImplementedError("C3b")

---# Part 4 — And the optimisation?Everything so far: we proved something, we measured it, the measurement agreed. This partchanges register. We will prove something, measure, and find that the theorem and themeasurement are **both correct and almost unrelated**.## T4 — theory *(corrected at the board)***T4.1** *(from Session 1)* Take $J(\theta) = \tfrac12\int_{-1}^{1}(\Phi_\theta - f)^2$ withtarget $f = \tanh$, and the one-neuron family $\Phi_{(c,w,b)}(x) = c\tanh(wx+b)$.Check that $\theta_+ = (1,1,0)$ and $\theta_- = (-1,-1,0)$ both realise $\tanh$, so$J(\theta_+) = J(\theta_-) = 0$. Compute $J$ at the midpoint, in closed form.*(You should find $1 - \tanh 1$.)* Conclude. Where does the obstruction live — in theactivation, or in the parametrisation?**T4.2** *(descent lemma)* Assume $\nabla J$ is $L$-Lipschitz, so that for all $\theta, \theta'$$$J(\theta') \;\le\; J(\theta) + \langle \nabla J(\theta), \theta' - \theta\rangle + \tfrac{L}{2}\|\theta' - \theta\|^2 .$$Put $\theta' = \theta_{k+1} = \theta_k - \eta \nabla J(\theta_k)$ and deduce$$J(\theta_{k+1}) \;\le\; J(\theta_k) - \eta\Big(1 - \tfrac{L\eta}{2}\Big)\|\nabla J(\theta_k)\|^2 .$$**T4.3** Take $\eta = 1/L$, sum over $k = 0, \dots, K-1$, and deduce **Theorem 3.2**:$$\min_{k < K} \|\nabla J(\theta_k)\| \;\le\; \sqrt{\frac{2L\big(J(\theta_0) - J^\star\big)}{K}} \;=\; O\big(K^{-1/2}\big).$$**T4.4** **The important one.** That is the only convergence guarantee this course has proved.Read it again and answer precisely: *what does it promise about the **value** of $J$ at theend?* Draw a one-dimensional landscape on which the theorem is satisfied and the minimiseris missed by any margin you like.**T4.5** *(Gauss–Newton)* Our loss is a sum of squares, $J(\theta) = \tfrac12\|r(\theta)\|^2$.Compute $\nabla J$ and $\nabla^2 J$. Near a good fit, $r \approx 0$: which term survives?Deduce the relation between $\kappa(\nabla^2 J)$ and $\kappa(Dr)$, and say what it implies fora first-order method **near the solution**. Which kind of method is the structural answer?**T4.6** Before running C4: Session 1 exhibited a network of $\Sigma_{128}(\tanh)$ with error$\sim 10^{-9}$, while training the same class returned $\sim 10^{-3}$. By what factor do youexpect a quasi-Newton phase to close that? **Write a number down.**

*Your notes on T4 — double-click to edit.*

## C4 — code: does a better optimiser close the gap?Same target as Lab 1, $f(x) = \sin(2\pi x) + \tfrac14 \sin(16\pi x)$, same class$\Sigma_n(\tanh)$ with $n = 128$. Three contenders:| | what it is ||---|---|| **convex solve** | freeze $(s_i, u_i)$ at random, solve for the outer coefficients in closed form. Convex. The error it returns is **achieved by a network we can write down**. || **Adam, 4000 steps** | train everything. Searches a strictly *larger* set. || **Adam 2000 + L-BFGS** | half the Adam budget, then a quasi-Newton phase. |The first function is given — it is not the point. You write the second.> ⏱ This cell takes about a minute. Start it, then read your notes on T4 while it runs.

In [ ]:
target = lambda z: torch.sin(2*np.pi*z) + 0.25*torch.sin(16*np.pi*z)xtr = torch.linspace(0, 1, 1000)ytr = target(xtr)def convex(n, seed):    """Freeze (s, u) at random, solve for c exactly. Linear least squares: convex,    closed form, no optimisation difficulty whatsoever. Returns the RMS error."""    g = torch.Generator().manual_seed(seed)    s = torch.randn(n, generator=g)*20.    u = torch.rand(n,  generator=g)    A = torch.cat([torch.tanh(s*(xtr.reshape(-1,1) - u)), torch.ones(len(xtr), 1)], 1)    c = torch.linalg.lstsq(A, ytr.reshape(-1,1)).solution    return ((A @ c).squeeze() - ytr).pow(2).mean().sqrt().item()def train(n, seed, adam_steps, lbfgs, lr=5e-3):    """Train ALL the parameters. Returns the RMS error."""    g  = torch.Generator().manual_seed(seed)    s  = (torch.randn(n, generator=g)*20.).requires_grad_(True)    u  = (torch.rand(n,  generator=g)).requires_grad_(True)    c  = (torch.randn(n, generator=g)/np.sqrt(n)).requires_grad_(True)    b0 = torch.zeros(1, requires_grad=True)    ps = [s, u, c, b0]    def loss():        return ((torch.tanh(s*(xtr.reshape(-1,1) - u)) @ c + b0) - ytr).pow(2).mean()    # TODO (1) run `adam_steps` steps of torch.optim.Adam(ps, lr=lr).    #          The usual three lines per step: zero_grad, backward, step.    #    # TODO (2) if lbfgs is True, follow with torch.optim.LBFGS(ps, max_iter=300,    #          history_size=40, tolerance_grad=1e-14, tolerance_change=1e-16,    #          line_search_fn="strong_wolfe").    #          L-BFGS needs a CLOSURE: a function that zeroes the gradient,    #          recomputes the loss, calls backward, and returns the loss.    #          Call lb.step(closure) four times.    #    # TODO (3) return the RMS error, i.e. loss().sqrt().item()    raise NotImplementedError("C4")# --- the comparison, once train() works ------------------------------------n, SEEDS = 128, range(5)rows = [("convex solve (features frozen)", [convex(n, s)                for s in SEEDS]),        ("Adam, 4000 steps",               [train(n, s, 4000, False)    for s in SEEDS]),        ("Adam 2000 + L-BFGS",             [train(n, s, 2000, True)     for s in SEEDS])]print(f"  {'':34} {'median':>10} {'min':>10} {'max':>10}")for name, v in rows:    print(f"  {name:<34} {np.median(v):>10.2e} {min(v):>10.2e} {max(v):>10.2e}")med = [np.median(v) for _, v in rows]plt.figure(figsize=(5.4, 3.0))plt.bar(range(3), med, color=[TEAL, WINE, OCHRE], width=.55)plt.errorbar(range(3), med,             yerr=[[m - min(v) for m, (_, v) in zip(med, rows)],                   [max(v) - m for m, (_, v) in zip(med, rows)]],             fmt="none", ecolor="#5C6663", capsize=3, lw=.9)plt.yscale("log"); plt.axhline(med[0], ls="--", lw=.9, color=TEAL)plt.xticks(range(3), ["convex", "Adam 4k", "Adam 2k\n+ L-BFGS"], fontsize=8)plt.ylabel("RMS error, $n=128$"); plt.grid(True, axis="y", ls=":", lw=.5)plt.tight_layout(); plt.show()

**T4.7** *(to discuss — the point of the whole session)*1. The first row is not a bound: it is a **network**, whose weights we could print. So a   network with that error demonstrably **exists** in $\Sigma_{128}(\tanh)$. What did   training return instead?2. L-BFGS beats Adam at **half** the Adam budget, and on **every** seed. Which prediction of   T4.5 is that, and is it confirmed quantitatively?3. Compare the surviving gap with the number you wrote down in T4.6.4. **Which of today's theorems is violated by that table?** Answer precisely.5. Someone objects: *"you simply did not tune it enough."* The objection is legitimate.   What exactly does the experiment establish, then — and what would you have to show to   change the conclusion?

---## What to take away1. **Differentiation is solved.** Exact, at every order, at a cost that does not grow with   the number of parameters. That is the only reason any of this is possible.2. **Differentiating in $x$ is not free.** A Laplacian costs $d$ sweeps, and in high dimension   that — not approximation theory — is the bill.3. **Optimisation is not solved**, and the one theorem we have promises something other than   what we want. Five orders of magnitude are sitting unexplained in the last table.*Session 3 writes the PINN method down properly. Session 6 is about those five orders.*---## To take home* **H1.** Redo T1.1–T1.4 for $y = c_2\tanh(w_2\tanh(w_1 x + b_1) + b_2)$ — two layers, five  parameters. Count the sweeps in both modes.* **H2.** In C4, add a fourth contender: Adam for $20\,000$ steps. Does five times the budget  buy as much as the quasi-Newton phase did? Compare *seed by seed*, not only the medians,  and say which of the two effects you would call real.* **H3.** Implement Hutchinson's estimator for $\Delta\Phi_\theta$ at one point in $d = 30$ and  compare its variance for Rademacher and Gaussian $v$ with the two formulas of T3.4. How  many samples before it beats the exact $d$-sweep computation in accuracy per unit time?* **H4.** *(open)* In C2 you found the bound is some two orders of magnitude above the  measured error. Build a function on which it is nearly attained. What does that function  have to look like?